# Imports and Setup

In [ ]:
%load_ext autoreload

import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s') # NOTSET, DEBUG, INFO, WARN, ERROR, CRITICAL

import os
import numpy as np

os.environ["JAX_PLATFORM_NAME"] = "cpu"  # must be set before importing jax

import jax
import jax.numpy as jnp
from jax import vmap, jit, grad, jacfwd

# Optional: higher numeric accuracy (do this before heavy JAX use)
# try:
#     from jax import config as jax_config
#     jax_config.update("jax_enable_x64", True)
# except Exception:
#     pass

import skewerjax as sjx

import matplotlib as mpl
from matplotlib.collections import LineCollection
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
plt.style.use('tableau-colorblind10')
font, rcnew = sjx.config.setup_matplotlib.matplotlib_default_config()
plt.rc('font', **font)
plt.rcParams.update(rcnew)
%matplotlib widget

In [ ]:
from skewerjax.skewers.skewers import SkewerBundle
from skewerjax.cosmology.cosmology import Cosmology

from skewerjax.fields.fields import ScalarField, VectorField, Fields
import skewerjax.fields.toy_model_samplers as toy_samplers
import skewerjax.utils.aux as aux
import skewerjax.utils.plotting_tools as pt

import skewerjax.optical_depth.optical_depth as od

from skewerjax.constants.physical_constants import phys

# Auxiliary objects that would be employed throughout the notebook

In [ ]:
x_axis_key = "velocities"
assert x_axis_key in ("wavelengths", "velocities"), "x_axis_key should be either 'wavelengths', or 'velocities'"

### Define radiation domination cosmology to employ in the different examples that require it

In [ ]:
# Create cosmology object
cosmology = Cosmology(H0=phys.H0, Omega_r_0=9.1436e-5, Omega_m_0=0.3089, Omega_k_0=0.0, Omega_lambda_0=0.6911)
# Define which cosmology to use, this is set as default to 0 since skewer and cosmology objects can deal with multiple cosmologies at once, 
# but other classes such as fields and optical depth are currently only set up for a single cosmology at a time
ii_cosmology = 0

### Define wavelength range and lines to study

In [ ]:
line_bundle = sjx.constants.lines.LineBundle(["HI-LyA", "SiIII-1206"]) # "SiII-1190", "SiII-1193"

### Auxiliary variables

In [ ]:
L_box = 300 # 51.7  # Mpc
z_box = 2.0

scale_factor_obs = 1.0

In [ ]:
scale_factor_right  = 1 / (1 + z_box)
scale_factor_left = float(cosmology.solve_a_of_x(
    ICs=jnp.array([1/(1+z_box)]),
    x_eval_Mpc=jnp.linspace(start=L_box, stop=0.0, num=2)
)[0, -1])

In [ ]:
T0_HI = 1e5
T0_Si = 2e6

n0_min = 1e-16
n0_max = 1e-10

def pk_template(k: jnp.ndarray, A, n, k0, k_cut):
    # P(k) = A (k/k0)^n exp[-(k/k_cut)^2]
    kk = jnp.maximum(k, 1e-12)
    return A * (kk / k0) ** n * jnp.exp(-(kk / k_cut) ** 2)

number_density_sampler = toy_samplers.number_density_sampler_gaussian_3d(
    key=jax.random.PRNGKey(0),
    pk_of_k=lambda k: pk_template(k, A=5.0, n=-2.5, k0=2.0, k_cut=50.0),
    n0_min=n0_min, n0_max=n0_max, L_box=L_box,
    grid_shape=(256, 256, 256), contrast=1.2,
)

In [ ]:
fields = Fields({
    "HI": {
        "temperature": ScalarField(toy_samplers.temperature_sampler(T0_HI, T0_HI, L_box)),    # UNITS: [K]
        "velocity": VectorField(toy_samplers.velocity_sampler(0.0, 0.0, L_box)),        # UNITS: [km/s]
        "number_density": ScalarField(number_density_sampler),                          # [cm^-3]
        "turbulence": ScalarField(toy_samplers.turbulence_sampler(b_turb=0.0)),         # UNITS: [km/s]
        "collisional": ScalarField(toy_samplers.collisional_sampler(A_coll=0.0))        # UNITS: [GHz]
    },
    "SiIII": {
        "temperature": ScalarField(toy_samplers.temperature_sampler(T0_Si, T0_Si, L_box)),    # UNITS: [K]
        "velocity": VectorField(toy_samplers.velocity_sampler(0.0, 0.0, L_box)),        # UNITS: [km/s]
        "number_density": ScalarField(number_density_sampler),                          # [cm^-3]
        "turbulence": ScalarField(toy_samplers.turbulence_sampler(b_turb=0.0)),         # UNITS: [km/s]
        "collisional": ScalarField(toy_samplers.collisional_sampler(A_coll=0.0))        # UNITS: [GHz]
    },
    # "SiII": {
    #     "temperature": ScalarField(toy_samplers.temperature_sampler(T0_Si, T0_Si, L_box)),    # UNITS: [K]
    #     "velocity": VectorField(toy_samplers.velocity_sampler(0.0, 0.0, L_box)),        # UNITS: [km/s]
    #     "number_density": ScalarField(number_density_sampler),                          # [cm^-3]
    #     "turbulence": ScalarField(toy_samplers.turbulence_sampler(b_turb=0.0)),         # UNITS: [km/s]
    #     "collisional": ScalarField(toy_samplers.collisional_sampler(A_coll=0.0))        # UNITS: [GHz]
    # },
})

In [ ]:
delta_lambda_limits = 1.0
min_lambda = jnp.min(line_bundle.lambda0_A)/scale_factor_right - delta_lambda_limits
max_lambda = jnp.max(line_bundle.lambda0_A)/scale_factor_left + delta_lambda_limits

resolution_lambda_A = 0.02 # in Angstroms
N_lambda = int((max_lambda - min_lambda) / resolution_lambda_A)

In [ ]:
lambda0_anchor_key = "Ly-\alpha"
if lambda0_anchor_key == "Ly-\alpha":
    lambda0_anchor = line_bundle.lines_for_species('HI')[0].lambda0_A
elif lambda0_anchor_key == None:
    lambda0_anchor = 1.0

In [ ]:
if x_axis_key == "wavelengths":
    x_label = r"$\lambda\ [\mathrm{\AA}]$"
    min_plot = min_lambda
    max_plot = max_lambda
    x_axis_array = jnp.linspace(min_plot, max_plot, N_lambda)  # in Angstroms
    wavelengths = x_axis_array
elif x_axis_key == "velocities":
    if lambda0_anchor_key == None:
        x_label = r"$c\, \ln\left(\lambda \left[\mathrm{\AA}\right]\right)\; \left[\mathrm{km\,/\,s}\right] $"
    else:
        x_label = r"$c\, \ln\left( \frac{\lambda}{\lambda_\dagger}\right)\; \left[\mathrm{km\,/\,s}\right]$"
    min_plot = phys.c*1e-3*jnp.log(min_lambda/lambda0_anchor)
    max_plot = phys.c*1e-3*jnp.log(max_lambda/lambda0_anchor)
    x_axis_array = jnp.linspace(min_plot, max_plot, N_lambda)
    wavelengths = lambda0_anchor * jnp.exp(x_axis_array/(phys.c*1e-3))

In [ ]:
save_path_global = "/home/STORAGE/SKEWERS/PLOTS/toy_metals/"
os.makedirs(save_path_global, exist_ok=True)

# Generate figures for gif

In [ ]:
N_ppsk = 1024
xx_LoS = jnp.linspace(start=0.0, stop=L_box, num=N_ppsk)
scale_factors = cosmology.solve_a_of_x(
    ICs=jnp.array([scale_factor_left]),
    x_eval_Mpc=xx_LoS
)[0]

N_grid_skewers = 3
points_grid_skewers = aux.make_skewer_grid_points(L_box, N_grid_skewers, N_ppsk, direction="y")

ic = N_grid_skewers//2
jc = N_grid_skewers//2

N_imshow = 1024
points_grid_imshow = aux.make_plane_points(L_box, 'x', L_box/2, N_imshow, N_imshow)
log10_number_density_imshow = np.log10(fields.number_density(jnp.concatenate([jnp.ones((N_imshow, N_imshow, 1)), points_grid_imshow], axis=-1)))

density_norm = mpl.colors.Normalize(
    vmin=float(np.min(log10_number_density_imshow)),
    vmax=float(np.max(log10_number_density_imshow)),
)

skewer_cmap = mpl.cm.get_cmap("gist_rainbow")
skewer_norm = mpl.colors.Normalize(vmin=0.0, vmax=float(N_ppsk - 1))

list_plot = range(1, N_ppsk, 8) # plot one each 2 N_ppsk
list_plot = [1, (N_ppsk-1)//2, N_ppsk-1]

if len(list_plot) < 5:
    show_plots = True
else:
    show_plots = False

for ii in list_plot:

    _N_ppsk = ii + 1
    _points_grid_skewers = points_grid_skewers[:, :, 0:_N_ppsk, :]

    _points = jnp.reshape(_points_grid_skewers, (N_grid_skewers*N_grid_skewers, _N_ppsk, 3))
    
    a0_per_skewer = scale_factors[_N_ppsk-1]
    x0_per_skewer = _points_grid_skewers[ic, jc, -1, 1]
    skewer_bundle = SkewerBundle(spatial_points=_points, cosmologies=cosmology, a0=a0_per_skewer, x0=x0_per_skewer)
    scale_factors_obs = jnp.repeat(scale_factor_obs, skewer_bundle.N_sk)
    optical_depth = od.compute_optical_depth(skewer_bundle, wavelengths, line_bundle, fields, scale_factors_obs) # Compute optical depth
    F_over_F0 = jnp.reshape(jnp.exp(-optical_depth), (line_bundle.n_lines, N_grid_skewers, N_grid_skewers, len(wavelengths)))

    points_central = skewer_bundle.spatial_points[ic * N_grid_skewers + jc]
    log10_number_density_central = np.log10(fields.number_density(jnp.concatenate([jnp.ones(points_central.shape[0:1])[..., None], points_central], axis=-1)))
    F_over_F0_central = F_over_F0[..., ic * N_grid_skewers + jc]

    fig1, axes1 = pt.plot_density_imshow_and_central_profile(
        points_central=np.asarray(points_central),
        points_grid_imshow=np.asarray(points_grid_imshow),
        log10_number_density_imshow=np.asarray(log10_number_density_imshow),
        log10_number_density_central=np.asarray(log10_number_density_central),
        line_bundle=line_bundle, scale_factor_start=scale_factor_left, scale_factor_end=a0_per_skewer,
        savepath_png=os.path.join(save_path_global, f"density_{ii}.png"),
        density_norm=density_norm,
        box_limits_yz=((0.0, L_box), (0.0, L_box)),
        box_limits_los=(0.0, L_box),
        skewer_cmap=skewer_cmap, skewer_norm=skewer_norm, show=show_plots,
    )
    
    fig2, axes2 = pt.plot_flux_panels(
        wavelengths=wavelengths, F_over_F0=np.asarray(F_over_F0),
        ic=ic, jc=jc, line_bundle=line_bundle,
        scale_factor_start=scale_factor_left, scale_factor_end=a0_per_skewer,
        x_axis_key=x_axis_key, min_x=min_plot, max_x=max_plot, lambda0_anchor=lambda0_anchor,
        savepath_png=os.path.join(save_path_global, f"flux_split_{ii}.png"),
        skewer_cmap=skewer_cmap, skewer_norm=skewer_norm,
        skewer_last_index=ii,  # "last" point shown in this iteration
        show=show_plots,
    )

    fig, ax = pt.plot_flux_panels_colored_by_los(
        wavelengths=wavelengths, F_over_F0=np.asarray(F_over_F0),
        ic=ic, jc=jc, line_bundle=line_bundle, x_axis_key=x_axis_key,
        scale_factor_start=scale_factor_left, scale_factor_end=a0_per_skewer,
        min_x=min_plot, max_x=max_plot, skewer_last_index=ii,
        lambda0_anchor=lambda0_anchor, skewer_cmap=skewer_cmap, skewer_norm=skewer_norm,
        savepath_png=os.path.join(save_path_global, f"flux_combined_{ii}.png"),
        show=show_plots,
    )

In [ ]:
linestyles = ["-", "--", "-.", ":", (0, (3, 1, 1, 1)), (0, (5, 1))]

fig, ax = plt.subplots(figsize=(10, 4.5))
legend_handles, legend_labels = [], []
for i_line, line in enumerate(line_bundle.lines):
    ls = linestyles[i_line % len(linestyles)]

    lam0 = float(line.lambda0_A)
    lam0_end = lam0 / scale_factor_right
    lam0_start = lam0 / scale_factor_left
    if x_axis_key == "wavelengths":
        x_end = lam0_end
        x_start = lam0_start
    elif x_axis_key == "velocities":
        x_end = phys.c*1e-3 * np.log(lam0_end / lambda0_anchor)
        x_start = phys.c*1e-3 * np.log(lam0_start / lambda0_anchor)
    ax.axvline(x_end, color="grey", linestyle=linestyles[i_line], lw=1.5, alpha=1.0)
    ax.axvline(x_start, color="grey", linestyle=linestyles[i_line], lw=1.5, alpha=1.0)

    ax.plot(x_axis_array, F_over_F0[i_line, ic, jc, :], color="royalblue", lw=1.5, linestyle=ls, zorder=2)

    name = getattr(line, "name", None) or getattr(line, "label", None) or f"Line {i_line}"
    legend_handles.append(Line2D([0], [0], color="royalblue", lw=2.2, linestyle=ls))
    legend_labels.append(name)

ax.set_xlim(float(np.min(x_axis_array)), float(np.max(x_axis_array)))
ax.set_ylim([-0.05, 1.05])

ax.set_xlabel(x_label, fontsize=20)
ax.set_ylabel(r"$e^{-\tau}$", fontsize=20)

ax.tick_params(which="both", direction="in", top=True, right=True, length=7, width=1.6, labelsize=18)
ax.tick_params(which="minor", length=4)
for spine in ax.spines.values():
    spine.set_linewidth(1.8)

ax.legend(
    legend_handles, legend_labels, loc="lower left", fontsize=16, handlelength=2.6, handletextpad=0.6, borderaxespad=0.6, ncols=1
)

fig.tight_layout()

savepath_png = os.path.join(save_path_global, "flux_separate_taus.png")
d = os.path.dirname(savepath_png)
if d:
    os.makedirs(d, exist_ok=True)
fig.savefig(savepath_png, dpi=200, bbox_inches="tight")

plt.show()

In [ ]:
optical_depth_total = jnp.sum(optical_depth, axis=0)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
for i_line, line in enumerate(line_bundle.lines):
    ls = linestyles[i_line % len(linestyles)]

    lam0 = float(line.lambda0_A)
    lam0_end = lam0 / scale_factor_right
    lam0_start = lam0 / scale_factor_left
    if x_axis_key == "wavelengths":
        x_end = lam0_end
        x_start = lam0_start
    elif x_axis_key == "velocities":
        x_end = phys.c*1e-3 * np.log(lam0_end / lambda0_anchor)
        x_start = phys.c*1e-3 * np.log(lam0_start / lambda0_anchor)
    ax.axvline(x_end, color="grey", linestyle=linestyles[i_line], lw=1.5, alpha=1.0)
    ax.axvline(x_start, color="grey", linestyle=linestyles[i_line], lw=1.5, alpha=1.0)

ax.plot(x_axis_array, jnp.exp(-optical_depth_total[ic * N_grid_skewers + jc, :]), color="royalblue", lw=1.5, linestyle='-', zorder=2)

ax.set_xlim(float(np.min(x_axis_array)), float(np.max(x_axis_array)))
ax.set_ylim([-0.05, 1.05])

ax.set_xlabel(x_label, fontsize=20)
ax.set_ylabel(r"$e^{-\tau}$", fontsize=20)

ax.tick_params(which="both", direction="in", top=True, right=True, length=7, width=1.6, labelsize=18)
ax.tick_params(which="minor", length=4)
for spine in ax.spines.values():
    spine.set_linewidth(1.8)

fig.tight_layout()

savepath_png = os.path.join(save_path_global, "flux_total.png")
d = os.path.dirname(savepath_png)
if d:
    os.makedirs(d, exist_ok=True)
fig.savefig(savepath_png, dpi=200, bbox_inches="tight")

plt.show()

# Generate grid

In [ ]:
N_grid_skewers = 8
N_ppsk = 1024
points_grid_skewers = aux.make_skewer_grid_points(L_box, N_grid_skewers, N_ppsk, direction="y")
points = jnp.reshape(points_grid_skewers, (N_grid_skewers*N_grid_skewers, N_ppsk, 3))
skewer_bundle = SkewerBundle(spatial_points=points, cosmologies=cosmology, a0=scale_factor_right, x0=L_box)

In [ ]:
delta_lambda_limits = 1.0
min_lambda = jnp.min(line_bundle.lambda0_A)/scale_factor_right - delta_lambda_limits
max_lambda = jnp.max(line_bundle.lambda0_A)/scale_factor_left + delta_lambda_limits

resolution_lambda_A = 0.07 # in Angstroms
N_lambda = int((max_lambda - min_lambda) / resolution_lambda_A)

In [ ]:
if x_axis_key == "wavelengths":
    x_label = r"$\lambda\ [\mathrm{\AA}]$"
    min_plot = min_lambda
    max_plot = max_lambda
    x_axis_array = jnp.linspace(min_plot, max_plot, N_lambda)  # in Angstroms
    wavelengths = x_axis_array
elif x_axis_key == "velocities":
    if lambda0_anchor_key == None:
        x_label = r"$c\, \ln\left(\lambda \left[\mathrm{\AA}\right]\right)\; \left[\mathrm{km\,/\,s}\right] $"
    else:
        x_label = r"$c\, \ln\left( \frac{\lambda}{\lambda_\dagger}\right)\; \left[\mathrm{km\,/\,s}\right]$"
    min_plot = phys.c*1e-3*jnp.log(min_lambda/lambda0_anchor)
    max_plot = phys.c*1e-3*jnp.log(max_lambda/lambda0_anchor)
    x_axis_array = jnp.linspace(min_plot, max_plot, N_lambda)
    wavelengths = lambda0_anchor * jnp.exp(x_axis_array/(phys.c*1e-3))

In [ ]:
array_size_bytes = N_lambda * skewer_bundle.N_sk * skewer_bundle.N_ppsk * line_bundle.n_lines * jnp.dtype(jnp.float64).itemsize
array_size_GB = array_size_bytes / 1e9
print(array_size_GB)

import psutil
memory_size = psutil.virtual_memory().total
memory_size_GB = memory_size / 1e9
print(memory_size_GB)

In [ ]:
scale_factors_obs = jnp.repeat(scale_factor_obs, skewer_bundle.N_sk)
optical_depth = od.compute_optical_depth(skewer_bundle, wavelengths, line_bundle, fields, scale_factors_obs) # Compute optical depth
F_over_F0 = jnp.exp(-optical_depth)

In [ ]:
# -----------------------
# Pick a subset of N skewers
# -----------------------
N_plot_sample = 4
seed = 0
rng = np.random.default_rng(seed)

N_lines, N_skewers, N_lambda = optical_depth.shape
idx = np.arange(N_skewers) if N_plot_sample >= N_skewers else rng.choice(N_skewers, size=N_plot_sample, replace=False)
idx = np.array(idx, dtype=int)
print("Selected skewer indices:", idx)

# colors: one color per selected skewer
cmap = plt.get_cmap("tab10")
colors = {ii: cmap(j % cmap.N) for j, ii in enumerate(idx)}

In [ ]:
linestyles = ["--", ":", "-.", (0, (3, 1, 1, 1)), (0, (5, 1))]

In [ ]:
list_handles = [Line2D([0], [0], color='k', lw=1.5, linestyle='-', label='Total')]
fig, axes = plt.subplots(N_plot_sample, 1, figsize=(14, 2*N_plot_sample), sharex=True, sharey=True)
for ii, _idx in enumerate(idx):
    ax = axes[ii]

    _F = np.prod(F_over_F0[:, _idx, :], axis=0)
    ax.plot(x_axis_array, _F, color=colors[_idx], lw=1.0, linestyle='-', alpha=1.0)
    
    for i_line, line in enumerate(line_bundle.lines):

        ax.plot(x_axis_array, F_over_F0[i_line, _idx, :], color=colors[_idx], lw=2.5, linestyle=linestyles[i_line])

        ls = linestyles[i_line % len(linestyles)]
            
        lam0 = float(line.lambda0_A)
        lam0_end = lam0 / scale_factor_right
        lam0_start = lam0 / scale_factor_left
        if x_axis_key == "wavelengths":
            x_end = lam0_end
            x_start = lam0_start
        elif x_axis_key == "velocities":
            x_end = phys.c*1e-3 * np.log(lam0_end / lambda0_anchor)
            x_start = phys.c*1e-3 * np.log(lam0_start / lambda0_anchor)
        ax.axvline(x_end, color="grey", linestyle=linestyles[i_line], lw=1.5, alpha=1.0)
        ax.axvline(x_start, color="grey", linestyle=linestyles[i_line], lw=1.5, alpha=1.0)

        if ii == 0:
            name = getattr(line, "name", None) or getattr(line, "label", None) or f"Line {i_line}"
            list_handles.append(Line2D([0], [0], color='k', lw=2.5, linestyle=ls, label=name))

    ax.set_ylim([-0.05, 1.05])
    ax.set_ylabel(r"$e^{-\tau}$", fontsize=20)
ax.set_xlabel(x_label, fontsize=20)

# custom legend
fig.legend(
    handles=list_handles, loc="upper center", bbox_to_anchor=(0.5, 0.96),    # push slightly above the figure
    ncols=len(line_bundle.lines) + 1, frameon=True, fontsize=16, handlelength=2.6, handletextpad=0.6,
)
savepath_png = os.path.join(save_path_global, "skewer_exsamples.png")
d = os.path.dirname(savepath_png)
if d:
    os.makedirs(d, exist_ok=True)
fig.savefig(savepath_png, dpi=200, bbox_inches="tight")

plt.show()

# $\xi1D$

In [ ]:
import skewerjax.summary_stats.xi1D as xi1D

In [ ]:
# 2PCF settings
n_bins = 256
r_min = 0.0
r_max = float(x_axis_array.max() - x_axis_array.min())*1.0
include_self = False
center = True
standardize = False
chunk_size = 1 << 15

In [ ]:
# 1) Precompute geometry ONCE
pb = xi1D.precompute_pair_binning_1d(
    x_axis_array, n_bins=n_bins, r_min=r_min, r_max=r_max, include_self=include_self, chunk_size=chunk_size,
)
r_centers = pb.r_centers

xi_auto = {}
xi_cross = {}          # only i<j stored here
xi_cross_diag = {}     # i==i stored here for validation

# 2) Auto for "total"
xi_auto["total"] = xi1D.two_point_corr_1d_from_precomp(
    pb, jnp.prod(F_over_F0, axis=0), center=center, standardize=standardize,
)

# 3) Auto for each line
for i, line in enumerate(line_bundle.lines):
    name = line.name
    xi_auto[name] = xi1D.two_point_corr_1d_from_precomp(
        pb, F_over_F0[i], center=center, standardize=standardize,
    )

# 4) Cross only for unique pairs (i<j), no repetition. Store as xi_cross[name_i][name_j] with i<j.
for i, line_i in enumerate(line_bundle.lines):
    name_i = line_i.name
    xi_cross[name_i] = {}
    
    for j in range(i + 1, len(line_bundle.lines)):
        name_j = line_bundle.lines[j].name
        xi_cross[name_i][name_j] = xi1D.two_point_corr_1d_cross_from_precomp(
            pb, F_over_F0[i], F_over_F0[j], center=center, standardize=standardize,
            symmetrize=True,  # this is ξ_{αX} symmetric definition
        )

In [ ]:
# quick lookup for rest wavelengths
line_by_name = {ln.name: ln for ln in line_bundle.lines}

# collect unique cross pairs (assumes xi_cross[name_i][name_j] only for i<j)
pairs = []
for name_i in xi_cross:
    for name_j in xi_cross[name_i]:
        pairs.append((name_i, name_j))

fig, ax = plt.subplots(figsize=(15, 7.0))

# -----------------------
# Color maps
# -----------------------
auto_names = [ln.name for ln in line_bundle.lines]

auto_cmap = plt.get_cmap("tab10")
auto_colors = {name: auto_cmap(i % 10) for i, name in enumerate(auto_names)}
total_color = "k"  # total stays black

n_pairs = max(len(pairs), 1)
cross_cmap = plt.get_cmap("tab20")
cross_colors = {pair: cross_cmap(t) for pair, t in zip(pairs, np.linspace(0.0, 1.0, n_pairs, endpoint=False))}

# -----------------------
# 1) Plot TOTAL (solid, black) mean ± std
# -----------------------
x_tot = np.asarray(xi_auto["total"])  # (N_samples, n_bins)
mean_tot = np.nanmean(x_tot, axis=0)
std_tot  = np.nanstd(x_tot, axis=0)

ax.plot(r_centers, mean_tot, lw=2.8, c=total_color, ls="-", alpha=1.0, zorder=5)
ax.fill_between(r_centers, mean_tot - std_tot, mean_tot + std_tot, color=total_color, alpha=0.12, zorder=4)

# -----------------------
# 2) Plot AUTO means (dotted, colored)
# -----------------------
auto_means = {}
for name in auto_names:
    x_auto = np.asarray(xi_auto[name])
    mean_auto = np.nanmean(x_auto, axis=0)
    auto_means[name] = mean_auto
    ax.plot(r_centers, mean_auto, lw=2.2, c=auto_colors[name], ls=":", alpha=0.95, zorder=3)

# -----------------------
# 3) Plot CROSS means (dashed, colored) + peak markers (same style/color)
# -----------------------
cross_means = []
for (name_i, name_j) in pairs:
    col = cross_colors[(name_i, name_j)]
    ls = "--"

    x_ij = np.asarray(xi_cross[name_i][name_j])
    mean_ij = np.nanmean(x_ij, axis=0)
    cross_means.append(mean_ij)

    # cross curve
    ax.plot(r_centers, mean_ij, lw=2.2, c=col, ls=ls, alpha=0.95, zorder=2)

    # expected peak marker for this pair
    lam_i = float(line_by_name[name_i].lambda0_A)
    lam_j = float(line_by_name[name_j].lambda0_A)

    if x_axis_key == "wavelengths":
        expected_cross_peak = abs(lam_i - lam_j) * (1 + z_box)
    elif x_axis_key == "velocities":
        expected_cross_peak = phys.c*1e-3 * np.log(lam_i / lam_j)
    else:
        expected_cross_peak = None

    if expected_cross_peak is not None:
        ax.axvline(float(np.abs(expected_cross_peak)), color=col, linestyle=ls, lw=1.6, alpha=0.55, zorder=1)

# -----------------------
# 4) Common y-limits (means + total envelope)
# -----------------------
ymins, ymaxs = [], []
ymins.append(np.nanmin(mean_tot - std_tot))
ymaxs.append(np.nanmax(mean_tot + std_tot))

for name in auto_names:
    ymins.append(np.nanmin(auto_means[name]))
    ymaxs.append(np.nanmax(auto_means[name]))

for mean_ij in cross_means:
    ymins.append(np.nanmin(mean_ij))
    ymaxs.append(np.nanmax(mean_ij))

ymin = float(np.nanmin(ymins))
ymax = float(np.nanmax(ymaxs))
pad = 0.06 * (ymax - ymin) if np.isfinite(ymax - ymin) and (ymax > ymin) else 1.0
ax.set_ylim(ymin - pad, ymax + pad)

# -----------------------
# 5) Cosmetics
# -----------------------
ax.axhline(0.0, color="k", lw=1.0, alpha=0.35)
ax.set_xlabel(r"Separation " + x_label)
ax.set_ylabel(r"$\xi$")

# =======================
# THREE LEGENDS
# =======================

# (1) Linestyle legend (what linestyle means)
style_handles = [
    Line2D([0], [0], color="k", lw=2.8, ls="-",  label=r"$\xi_\mathrm{T}$"),
    Line2D([0], [0], color="k", lw=2.2, ls=":",  label=r"$\xi_\mathrm{i}$"),
    Line2D([0], [0], color="k", lw=2.2, ls="--", label=r"$\xi_\mathrm{i,j}$"),
]
leg_style = ax.legend(handles=style_handles, loc="upper left", frameon=True, fontsize=16, ncols=3)
ax.add_artist(leg_style)

# (2) Auto legend: colors only, includes total + each auto line
auto_color_handles = [Line2D([0], [0], color=total_color, lw=5.0, ls="-", label=r"$\xi_\mathrm{total}$")]
for name in auto_names:
    auto_color_handles.append(Line2D([0], [0], color=auto_colors[name], lw=5.0, ls="-", label=rf"$\xi_\mathrm{{{name}}}$"))

leg_auto = ax.legend(handles=auto_color_handles, loc="lower right", frameon=True, fontsize=16, ncols=len(auto_names)+1)
ax.add_artist(leg_auto)

# (3) Cross legend: colors only
cross_color_handles = []
for (name_i, name_j) in pairs:
    cross_color_handles.append(
        Line2D([0], [0], color=cross_colors[(name_i, name_j)], lw=5.0, ls="-", label=rf"$\xi_\mathrm{{{name_i},{name_j}}}$")
    )

# Place cross legend lower right to avoid collisions (tweak as you like)
if len(cross_color_handles) > 0:
    leg_cross = ax.legend(handles=cross_color_handles, loc="upper right", frameon=True, fontsize=16, ncol=1)
    ax.add_artist(leg_cross)

# -----------------------
# 6) Save + show
# -----------------------
savepath_png = os.path.join(save_path_global, "xi1D.png")
os.makedirs(os.path.dirname(savepath_png), exist_ok=True)
fig.savefig(savepath_png, dpi=200, bbox_inches="tight")

plt.tight_layout()
plt.show()

# P1D

In [ ]:
import skewerjax.summary_stats.P1D as P1D

In [ ]:
%autoreload

In [ ]:
# 1) Precompute Welch geometry ONCE
pc = P1D.precompute_welch_1d(
    x_axis_array,
    nperseg=4096,
    noverlap=256,
    window="hann",
    detrend="constant",
    scaling="density",
    average="mean",
    as_angular=True,
)
k = None  # will be returned by the functions below

P_auto = {}
P_cross = {}

# 2) Auto for "total"
k, P_auto["total"] = P1D.psd_auto_from_precomp(pc, jnp.prod(F_over_F0, axis=0))

# 3) Auto for each line
for i, line in enumerate(line_bundle.lines):
    name = line.name
    _, P_auto[name] = P1D.psd_auto_from_precomp(pc, F_over_F0[i])

# 4) Cross only for unique pairs (i<j)
for i, line_i in enumerate(line_bundle.lines):
    name_i = line_i.name
    P_cross[name_i] = {}
    for j in range(i + 1, len(line_bundle.lines)):
        name_j = line_bundle.lines[j].name
        _, P_cross[name_i][name_j] = P1D.csd_cross_from_precomp(
            pc, F_over_F0[i], F_over_F0[j],
            symmetrize=False,   # keep complex cross-spectrum (recommended)
        )

# Optional: coherence for a given pair
# _, Pxx = psd_auto_from_precomp(pc, F_over_F0[i])
# _, Pyy = psd_auto_from_precomp(pc, F_over_F0[j])
# _, Pxy = csd_cross_from_precomp(pc, F_over_F0[i], F_over_F0[j], symmetrize=False)
# coh = coherence_from_psd_csd(Pxx, Pyy, Pxy)

In [ ]:
# quick lookup for rest wavelengths
line_by_name = {ln.name: ln for ln in line_bundle.lines}

# collect unique cross pairs (assumes P_cross[name_i][name_j] only for i<j)
pairs = []
for name_i in P_cross:
    for name_j in P_cross[name_i]:
        pairs.append((name_i, name_j))

fig, ax = plt.subplots(figsize=(15, 7.0))

# -----------------------
# 0) k-axis (drop k=0 for log-x, cleaner spectra)
# -----------------------
k_arr = np.asarray(k)  # shape (N_k,)
if k_arr[0] == 0.0:
    k_plot = k_arr[1:]
    sl = slice(1, None)
else:
    k_plot = k_arr
    sl = slice(None)

# -----------------------
# Color maps
# -----------------------
auto_names = [ln.name for ln in line_bundle.lines]

auto_cmap = plt.get_cmap("tab10")
auto_colors = {name: auto_cmap(i % 10) for i, name in enumerate(auto_names)}
total_color = "k"  # total stays black

n_pairs = max(len(pairs), 1)
cross_cmap = plt.get_cmap("tab20")
cross_colors = {
    pair: cross_cmap(t)
    for pair, t in zip(pairs, np.linspace(0.0, 1.0, n_pairs, endpoint=False))
}

# -----------------------
# 1) Plot TOTAL (solid, black) mean ± std
# -----------------------
P_tot = np.asarray(P_auto["total"])[:, sl]  # (N_samples, N_k-1)
mean_tot = np.nanmean(P_tot, axis=0)
std_tot  = np.nanstd(P_tot, axis=0)

ax.plot(k_plot, mean_tot, lw=2.8, c=total_color, ls="-", alpha=1.0, zorder=5)
ax.fill_between(k_plot, mean_tot - std_tot, mean_tot + std_tot, color=total_color, alpha=0.12, zorder=4)

# -----------------------
# 2) Plot AUTO means (dotted, colored)
# -----------------------
auto_means = {}
for name in auto_names:
    P_auto_i = np.asarray(P_auto[name])[:, sl]
    mean_i = np.nanmean(P_auto_i, axis=0)
    auto_means[name] = mean_i
    ax.plot(k_plot, mean_i, lw=2.2, c=auto_colors[name], ls=":", alpha=0.95, zorder=3)

# -----------------------
# 3) Plot CROSS means (dashed, colored) + optional k-scale markers
# -----------------------
cross_means = []
for (name_i, name_j) in pairs:
    col = cross_colors[(name_i, name_j)]
    ls = "--"

    P_ij = np.asarray(P_cross[name_i][name_j])[:, sl]
    mean_ij = np.nanmean(P_ij, axis=0)
    cross_means.append(mean_ij)

    # cross curve
    ax.plot(k_plot, mean_ij, lw=2.2, c=col, ls=ls, alpha=0.95, zorder=2)

    # --- optional reference scale marker: k_scale = 2π / expected_separation
    lam_i = float(line_by_name[name_i].lambda0_A)
    lam_j = float(line_by_name[name_j].lambda0_A)

    expected_sep = None
    if x_axis_key == "wavelengths":
        # separation in observed wavelength units (same you used in xi plot)
        expected_sep = abs(lam_i - lam_j) * (1 + z_box)
    elif x_axis_key == "velocities":
        # if your x-axis is in velocity units (km/s) and matches this convention:
        # expected_sep = c * ln(lam_i/lam_j)
        expected_sep = phys.c * 1e-3 * np.log(lam_i / lam_j)

    if expected_sep is not None and expected_sep != 0:
        k_scale = 2.0 * np.pi / abs(expected_sep)
        # marker is a "scale" guide, not a guaranteed peak in P(k)
        ax.axvline(float(k_scale), color=col, linestyle=ls, lw=1.6, alpha=0.45, zorder=1)

# -----------------------
# 4) y-scale + limits (auto is positive, cross can be negative)
# -----------------------
all_vals = [mean_tot - std_tot, mean_tot + std_tot]
all_vals += [auto_means[name] for name in auto_names]
all_vals += cross_means

finite = np.concatenate([v[np.isfinite(v)] for v in all_vals if v is not None])
has_negative = np.any(finite < 0)

# x is almost always log for power spectra
ax.set_xscale("log")

if has_negative:
    # signed cross-power -> use symlog
    # choose linthresh from a small percentile scale of |finite|
    abs_f = np.abs(finite)
    linthresh = float(np.nanpercentile(abs_f, 5)) if abs_f.size else 1e-6
    linthresh = max(linthresh, 1e-12)
    ax.set_yscale("symlog", linthresh=linthresh)
else:
    # all positive -> use log y
    ax.set_yscale("log")

# y-limits (robust, symmetric if symlog)
if finite.size:
    ymin = float(np.nanmin(finite))
    ymax = float(np.nanmax(finite))
    if has_negative:
        m = max(abs(ymin), abs(ymax))
        ax.set_ylim(-1.15 * m, 1.15 * m)
    else:
        # pad multiplicatively for log
        ax.set_ylim(max(ymin * 0.8, 1e-30), ymax * 1.25)

# -----------------------
# 5) Cosmetics
# -----------------------
ax.axhline(0.0, color="k", lw=1.0, alpha=0.25)
# ax.set_xlabel(r"Wavenumber " + k_label)  # e.g. r"$k$ [1/Mpc]" or similar
ax.set_ylabel(r"$P_{1\mathrm{D}}(k)$")

# =======================
# THREE LEGENDS (same structure as your xi plot)
# =======================

# (1) Linestyle legend (what linestyle means)
style_handles = [
    Line2D([0], [0], color="k", lw=2.8, ls="-",  label=r"$P_\mathrm{T}$"),
    Line2D([0], [0], color="k", lw=2.2, ls=":",  label=r"$P_\mathrm{i}$"),
    Line2D([0], [0], color="k", lw=2.2, ls="--", label=r"$P_{\mathrm{i,j}}$"),
]
leg_style = ax.legend(handles=style_handles, loc="upper left", frameon=True, fontsize=16, ncols=3)
ax.add_artist(leg_style)

# (2) Auto legend: colors only, includes total + each auto line
auto_color_handles = [Line2D([0], [0], color=total_color, lw=5.0, ls="-", label=r"$P_\mathrm{total}$")]
for name in auto_names:
    auto_color_handles.append(
        Line2D([0], [0], color=auto_colors[name], lw=5.0, ls="-", label=rf"$P_\mathrm{{{name}}}$")
    )
leg_auto = ax.legend(handles=auto_color_handles, loc="lower right", frameon=True, fontsize=16, ncols=len(auto_names)+1)
ax.add_artist(leg_auto)

# (3) Cross legend: colors only
cross_color_handles = []
for (name_i, name_j) in pairs:
    cross_color_handles.append(
        Line2D([0], [0], color=cross_colors[(name_i, name_j)], lw=5.0, ls="-",
               label=rf"$P_\mathrm{{{name_i},{name_j}}}$")
    )
if len(cross_color_handles) > 0:
    leg_cross = ax.legend(handles=cross_color_handles, loc="upper right", frameon=True, fontsize=16, ncol=1)
    ax.add_artist(leg_cross)

# # -----------------------
# # 6) Save + show
# # -----------------------
# savepath_png = os.path.join(save_path_global, "P1D.png")
# os.makedirs(os.path.dirname(savepath_png), exist_ok=True)
# fig.savefig(savepath_png, dpi=200, bbox_inches="tight")

# plt.tight_layout()
# plt.show()

In [ ]:
# quick lookup for rest wavelengths
line_by_name = {ln.name: ln for ln in line_bundle.lines}

# collect unique cross pairs (assumes P_cross[name_i][name_j] only for i<j)
pairs = []
for name_i in P_cross:
    for name_j in P_cross[name_i]:
        pairs.append((name_i, name_j))

fig, ax = plt.subplots(figsize=(15, 7.0))

# -----------------------
# 0) k-axis (drop k=0 for log-x, cleaner spectra)
# -----------------------
k_arr = np.asarray(k)  # shape (N_k,)
if k_arr[0] == 0.0:
    k_plot = k_arr[1:]
    sl = slice(1, None)
else:
    k_plot = k_arr
    sl = slice(None)

# -----------------------
# 0b) Choose reference (Lyα) for normalization
# -----------------------
# Try common names; fall back to first line if not found.
lya_candidates = ["Lyalpha", "Lyα", "Lya", "HI1215", "H I 1215", "lya", "lyalpha"]
auto_names = [ln.name for ln in line_bundle.lines]

lya_name = next((n for n in auto_names if n in lya_candidates), None)
if lya_name is None:
    # also try case-insensitive match
    low_map = {n.lower(): n for n in auto_names}
    lya_name = next((low_map.get(c.lower()) for c in lya_candidates if c.lower() in low_map), None)

if lya_name is None:
    lya_name = auto_names[0]  # safe fallback

# Reference spectrum: mean Lyα auto (avoid zeros)
P_lya = np.asarray(P_auto[lya_name])[:, sl]                 # (N_samples, N_k')
P_lya_mean = np.nanmean(P_lya, axis=0)                      # (N_k',)
eps_ref = 1e-30
P_ref = np.where(np.isfinite(P_lya_mean) & (np.abs(P_lya_mean) > eps_ref), P_lya_mean, np.nan)

# -----------------------
# Color maps
# -----------------------
auto_cmap = plt.get_cmap("tab10")
auto_colors = {name: auto_cmap(i % 10) for i, name in enumerate(auto_names)}
total_color = "k"  # total stays black

n_pairs = max(len(pairs), 1)
cross_cmap = plt.get_cmap("tab20")
cross_colors = {
    pair: cross_cmap(t)
    for pair, t in zip(pairs, np.linspace(0.0, 1.0, n_pairs, endpoint=False))
}

# -----------------------
# 1) Plot TOTAL (solid, black) mean ± std, normalized by Lyα auto mean
# -----------------------
P_tot = np.asarray(P_auto["total"])[:, sl]  # (N_samples, N_k')
mean_tot = np.nanmean(P_tot, axis=0) / P_ref
std_tot  = np.nanstd(P_tot, axis=0)  / P_ref

ax.plot(k_plot, mean_tot, lw=2.8, c=total_color, ls="-", alpha=1.0, zorder=5)
ax.fill_between(k_plot, mean_tot - std_tot, mean_tot + std_tot, color=total_color, alpha=0.12, zorder=4)

# -----------------------
# 2) Plot AUTO means (dotted, colored), normalized
# -----------------------
auto_means = {}
for name in auto_names:
    P_auto_i = np.asarray(P_auto[name])[:, sl]
    mean_i = np.nanmean(P_auto_i, axis=0) / P_ref
    auto_means[name] = mean_i
    ax.plot(k_plot, mean_i, lw=2.2, c=auto_colors[name], ls=":", alpha=0.95, zorder=3)

# -----------------------
# 3) Plot CROSS means (dashed, colored), normalized
#     NOTE: if P_cross is complex, decide how to show it:
#           - Jonas likely wants relative amplitudes -> show Re(P_ij)/P_lya or |P_ij|/P_lya.
#           Here we plot Re(P_ij) by default (keeps sign), but you can swap to np.abs(...)
# -----------------------
cross_means = []
for (name_i, name_j) in pairs:
    col = cross_colors[(name_i, name_j)]
    ls = "--"

    P_ij = np.asarray(P_cross[name_i][name_j])[:, sl]
    mean_ij = np.nanmean(P_ij, axis=0)

    # choose representation for cross (uncomment the one you want)
    mean_ij_plot = np.real(mean_ij)          # signed cross-power (recommended if you care about sign)
    # mean_ij_plot = np.abs(mean_ij)         # magnitude-only (recommended if you care about amplitude only)

    mean_ij_plot = mean_ij_plot / P_ref
    cross_means.append(mean_ij_plot)

    ax.plot(k_plot, mean_ij_plot, lw=2.2, c=col, ls=ls, alpha=0.95, zorder=2)

# -----------------------
# 4) Axes: x in log, y in linear (Jonas request)
# -----------------------
ax.set_xscale("log")
ax.set_yscale("linear")

# Robust y-limits (linear): use percentiles to avoid a single wild point
all_vals = [mean_tot - std_tot, mean_tot + std_tot]
all_vals += [auto_means[name] for name in auto_names]
all_vals += cross_means

finite = np.concatenate([v[np.isfinite(v)] for v in all_vals if v is not None])
if finite.size:
    lo = float(np.nanpercentile(finite, 1))
    hi = float(np.nanpercentile(finite, 99))
    # pad a bit
    pad = 0.1 * max(abs(lo), abs(hi), 1e-12)
    ax.set_ylim(lo - pad, hi + pad)

# -----------------------
# 5) Cosmetics
# -----------------------
ax.axhline(0.0, color="k", lw=1.0, alpha=0.25)
ax.set_ylabel(r"$P_{1\mathrm{D}}(k)\,/\,P^{\mathrm{Ly}\alpha}_{1\mathrm{D}}(k)$")
ax.set_title(rf"P1D normalized by {lya_name} auto (mean)")

# =======================
# THREE LEGENDS (same structure as your xi plot)
# =======================

# (1) Linestyle legend (what linestyle means)
style_handles = [
    Line2D([0], [0], color="k", lw=2.8, ls="-",  label=r"$P_\mathrm{T}$"),
    Line2D([0], [0], color="k", lw=2.2, ls=":",  label=r"$P_\mathrm{i}$"),
    Line2D([0], [0], color="k", lw=2.2, ls="--", label=r"$P_{\mathrm{i,j}}$"),
]
leg_style = ax.legend(handles=style_handles, loc="upper left", frameon=True, fontsize=16, ncols=3)
ax.add_artist(leg_style)

# (2) Auto legend: colors only, includes total + each auto line
auto_color_handles = [Line2D([0], [0], color=total_color, lw=5.0, ls="-", label=r"$P_\mathrm{total}$")]
for name in auto_names:
    auto_color_handles.append(
        Line2D([0], [0], color=auto_colors[name], lw=5.0, ls="-", label=rf"$P_\mathrm{{{name}}}$")
    )
leg_auto = ax.legend(handles=auto_color_handles, loc="lower right", frameon=True, fontsize=16, ncols=min(len(auto_names)+1, 6))
ax.add_artist(leg_auto)

# (3) Cross legend: colors only
cross_color_handles = []
for (name_i, name_j) in pairs:
    cross_color_handles.append(
        Line2D([0], [0], color=cross_colors[(name_i, name_j)], lw=5.0, ls="-",
               label=rf"$P_\mathrm{{{name_i},{name_j}}}$")
    )
if len(cross_color_handles) > 0:
    leg_cross = ax.legend(handles=cross_color_handles, loc="upper right", frameon=True, fontsize=16, ncol=1)
    ax.add_artist(leg_cross)

# plt.tight_layout()
# plt.show()

# Periodic boundary conditions

In [ ]:
v_anchor = phys.c*1e-3 * np.log(scale_factor_obs / scale_factor_right)
delta_v_box = phys.c*1e-3 * jnp.log(scale_factor_right/scale_factor_left)
x_axis_array_anchor = x_axis_array - v_anchor

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(15, 4))
colors = plt.cm.tab10(np.linspace(0, 1, len(line_bundle.lines)))
_idx = 0
for i_line, line in enumerate(line_bundle.lines):
    ax.plot(x_axis_array_anchor, F_over_F0[i_line, _idx, :], color=colors[i_line], lw=1.5, linestyle='-', alpha=1.0)
    x_left = phys.c*1e-3 * jnp.log(line.lambda0_A / lambda0_anchor)
    x_right = x_left + delta_v_box
    ax.axvline(x_left, color=colors[i_line], linestyle='--', lw=2.0, alpha=1.0)
    ax.axvline(x_right, color=colors[i_line], linestyle='--', lw=2.0, alpha=1.0)
ax.set_xlim(-2700, delta_v_box+500)
fig.tight_layout()
plt.show()

In [ ]:
def wrap_flux_lines_all_skewers(
    x_axis_array_anchor,
    F_over_F0,
    line_bundle,
    lambda0_anchor,
    delta_v_box,
):
    """
    Periodic wrapping of flux lines for all skewers.

    Parameters
    ----------
    x_axis_array_anchor : array, shape (n_pix,)
    F_over_F0 : array, shape (n_lines, n_skewers, n_pix)

    Returns
    -------
    x_axis_array_wrapped : dict[line_key] -> array, shape (n_keep,)
    F_over_F0_wrapped    : dict[line_key] -> array, shape (n_skewers, n_keep)
    line_intervals       : dict[line_key] -> metadata
    """
    x_axis_array_anchor = jnp.asarray(x_axis_array_anchor)
    F_over_F0 = jnp.asarray(F_over_F0)

    x_axis_array_wrapped = {}
    F_over_F0_wrapped = {}
    line_intervals = {}

    for i_line, line in enumerate(line_bundle.lines):
        line_key = str(line.name)

        x_left = phys.c * 1e-3 * jnp.log(line.lambda0_A / lambda0_anchor)
        x_right = x_left + delta_v_box

        # Same physical selection as in your original function
        mask = (x_axis_array_anchor >= x_left) & (x_axis_array_anchor < x_right)
        idx = jnp.where(mask)[0]

        x_crop = x_axis_array_anchor[idx]                         # (n_keep,)
        flux_crop = jnp.take(F_over_F0[i_line], idx, axis=-1)     # (n_skewers, n_keep)

        x_wrap = jnp.mod(x_crop, delta_v_box)
        order = jnp.argsort(x_wrap)

        x_axis_array_wrapped[line_key] = x_wrap[order]                  # (n_keep,)
        F_over_F0_wrapped[line_key] = jnp.take(flux_crop, order, axis=-1)  # (n_skewers, n_keep)

        line_intervals[line_key] = {
            "x_left": x_left,
            "x_right": x_right,
            "x_left_wrapped": jnp.mod(x_left, delta_v_box),
            "n_keep": len(idx),
        }

    return x_axis_array_wrapped, F_over_F0_wrapped, line_intervals

In [ ]:
x_axis_array_wrapped, F_over_F0_wrapped, line_intervals = wrap_flux_lines_all_skewers(
    x_axis_array_anchor=x_axis_array_anchor, F_over_F0=F_over_F0,
    line_bundle=line_bundle, lambda0_anchor=lambda0_anchor, delta_v_box=delta_v_box,
)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(15, 4))
colors = plt.cm.tab10(np.linspace(0, 1, len(line_bundle.lines)))
_idx = 0
for i_line, line in enumerate(line_bundle.lines):
    ax.plot(x_axis_array_wrapped[line.name], F_over_F0_wrapped[line.name][_idx, :], color=colors[i_line], lw=1.5, linestyle='-', alpha=1.0)
ax.set_xlim(-2700, delta_v_box+500)
fig.tight_layout()
plt.show()

In [ ]:
def compute_total_wrapped_flux_all_skewers(
    x_axis_array_wrapped,
    F_over_F0_wrapped,
    x_common=None,
    n_pix_common=None,
):
    """
    Interpolate all wrapped line fluxes onto a common x-axis and compute
    the total transmitted flux by multiplying the individual contributions.

    Parameters
    ----------
    x_axis_array_wrapped : dict[line_key] -> array, shape (n_pix_line,)
        Wrapped x-coordinates for each line, already sorted in [0, delta_v_box).

    F_over_F0_wrapped : dict[line_key] -> array, shape (n_skewers, n_pix_line)
        Wrapped transmitted flux for each line.

    x_common : array, optional
        Common x-axis where all lines will be interpolated. If None, it is built
        automatically as a uniform grid over the full wrapped domain.

    n_pix_common : int, optional
        Number of pixels for the automatic common grid if x_common is None.
        If None, uses the maximum number of pixels among all wrapped lines.

    Returns
    -------
    x_common : array, shape (n_pix_common,)
        Common wrapped x-axis.

    F_interp_dict : dict[line_key] -> array, shape (n_skewers, n_pix_common)
        Each line interpolated onto x_common.

    F_total : array, shape (n_skewers, n_pix_common)
        Total transmitted flux, i.e. product over all lines.
    """
    keys = list(x_axis_array_wrapped.keys())
    if len(keys) == 0:
        raise ValueError("No wrapped lines were provided.")

    # Infer common domain size from wrapped coordinates
    xmax = max(float(jnp.max(x_axis_array_wrapped[key])) for key in keys)

    if x_common is None:
        if n_pix_common is None:
            n_pix_common = max(len(x_axis_array_wrapped[key]) for key in keys)
        x_common = jnp.linspace(0.0, xmax, n_pix_common)

    x_common = jnp.asarray(x_common)

    F_interp_dict = {}
    F_total = None

    for key in keys:
        x_line = jnp.asarray(x_axis_array_wrapped[key])               # (n_line_pix,)
        F_line = jnp.asarray(F_over_F0_wrapped[key])                 # (n_skewers, n_line_pix)

        # Interpolate each skewer independently onto x_common
        F_interp = jnp.stack(
            [jnp.interp(x_common, x_line, F_line[i_sk]) for i_sk in range(F_line.shape[0])],
            axis=0,
        )  # (n_skewers, n_pix_common)

        F_interp_dict[key] = F_interp

        if F_total is None:
            F_total = F_interp
        else:
            F_total = F_total * F_interp

    return x_common, F_interp_dict, F_total

In [ ]:
x_common, F_interp_dict, F_total = compute_total_wrapped_flux_all_skewers(
    x_axis_array_wrapped=x_axis_array_wrapped,
    F_over_F0_wrapped=F_over_F0_wrapped,
    n_pix_common=2048,   # optional
)

In [ ]:
# -----------------------
# Pick a subset of N skewers
# -----------------------
N_plot_sample = 4
seed = 0
rng = np.random.default_rng(seed)

N_lines, N_skewers, N_lambda = optical_depth.shape
idx = np.arange(N_skewers) if N_plot_sample >= N_skewers else rng.choice(N_skewers, size=N_plot_sample, replace=False)
idx = np.array(idx, dtype=int)
print("Selected skewer indices:", idx)

# colors: one color per selected skewer
cmap = plt.get_cmap("tab10")
colors = {ii: cmap(j % cmap.N) for j, ii in enumerate(idx)}

In [ ]:
list_handles = [Line2D([0], [0], color='k', lw=1.5, linestyle='-', label='Total')]

fig, axes = plt.subplots(
    N_plot_sample, 1,
    figsize=(14, 2 * N_plot_sample),
    sharex=True,
    sharey=True,
)

if N_plot_sample == 1:
    axes = [axes]

wrapped_keys = list(F_interp_dict.keys())

for ii, _idx in enumerate(idx):
    ax = axes[ii]

    # Total wrapped flux
    ax.plot(
        x_common,
        F_total[_idx, :],
        color=colors[_idx],
        lw=1.0,
        linestyle='-',
        alpha=1.0,
    )

    # Individual wrapped line contributions
    for i_line, key in enumerate(wrapped_keys):
        ls = linestyles[i_line % len(linestyles)]

        ax.plot(
            x_common,
            F_interp_dict[key][_idx, :],
            color=colors[_idx],
            lw=2.5,
            linestyle=ls,
            alpha=1.0,
        )

        # Wrapped left-edge marker for this line
        ax.axvline(
            line_intervals[key]["x_left_wrapped"],
            color="grey",
            linestyle=ls,
            lw=1.5,
            alpha=1.0,
        )

        if ii == 0:
            list_handles.append(
                Line2D([0], [0], color='k', lw=2.5, linestyle=ls, label=key)
            )

    ax.set_ylim([-0.05, 1.05])
    ax.set_xlim([0, delta_v_box])
    ax.set_ylabel(r"$e^{-\tau}$", fontsize=20)

ax.set_xlabel(r"$\Delta v \; [\mathrm{km/s}]$", fontsize=20)


plt.show()

# $\xi1D$

In [ ]:
# 2PCF settings
n_bins = 256
r_min = 0.0
r_max = float(delta_v_box)*1.0
include_self = False
center = True
standardize = False
chunk_size = 1 << 15

In [ ]:
# 1) Precompute geometry ONCE on the common wrapped grid
pb = xi1D.precompute_pair_binning_1d(
    x_common,
    n_bins=n_bins,
    r_min=r_min,
    r_max=r_max,
    include_self=include_self,
    chunk_size=chunk_size,
)
r_centers = pb.r_centers

xi_auto = {}
xi_cross = {}       # only i<j stored here
xi_cross_diag = {}  # optional: auto computed through cross routine for validation

wrapped_keys = list(F_interp_dict.keys())

# 2) Auto for total wrapped flux
xi_auto["total"] = xi1D.two_point_corr_1d_from_precomp(
    pb,
    F_total,
    center=center,
    standardize=standardize,
)

# 3) Auto for each wrapped line
for key in wrapped_keys:
    xi_auto[key] = xi1D.two_point_corr_1d_from_precomp(
        pb,
        F_interp_dict[key],
        center=center,
        standardize=standardize,
    )

# 4) Cross only for unique pairs (i < j)
for i, key_i in enumerate(wrapped_keys):
    xi_cross[key_i] = {}

    for j in range(i + 1, len(wrapped_keys)):
        key_j = wrapped_keys[j]

        xi_cross[key_i][key_j] = xi1D.two_point_corr_1d_cross_from_precomp(
            pb,
            F_interp_dict[key_i],
            F_interp_dict[key_j],
            center=center,
            standardize=standardize,
            symmetrize=True,
        )

# 5) Optional: diagonal cross-checks, useful for validation
for key in wrapped_keys:
    xi_cross_diag[key] = xi1D.two_point_corr_1d_cross_from_precomp(
        pb,
        F_interp_dict[key],
        F_interp_dict[key],
        center=center,
        standardize=standardize,
        symmetrize=True,
    )

In [ ]:
# -------------------------------------------------
# Build mapping between wrapped keys and line objects
# -------------------------------------------------
wrapped_keys = list(F_interp_dict.keys())
key_to_line = {key: line for key, line in zip(wrapped_keys, line_bundle.lines)}
key_to_name = {
    key: getattr(line, "name", None) or getattr(line, "label", None) or key
    for key, line in key_to_line.items()
}

# collect unique cross pairs (assumes xi_cross[key_i][key_j] only for i<j)
pairs = []
for key_i in xi_cross:
    for key_j in xi_cross[key_i]:
        pairs.append((key_i, key_j))

fig, ax = plt.subplots(figsize=(15, 7.0))

# -----------------------
# Color maps
# -----------------------
auto_keys = wrapped_keys

auto_cmap = plt.get_cmap("tab10")
auto_colors = {key: auto_cmap(i % 10) for i, key in enumerate(auto_keys)}
total_color = "k"

n_pairs = max(len(pairs), 1)
cross_cmap = plt.get_cmap("tab20")
cross_colors = {
    pair: cross_cmap(t)
    for pair, t in zip(pairs, np.linspace(0.0, 1.0, n_pairs, endpoint=False))
}

# -----------------------
# 1) Plot TOTAL (solid, black) mean ± std
# -----------------------
x_tot = np.asarray(xi_auto["total"])   # (N_skewers, n_bins)
mean_tot = np.nanmean(x_tot, axis=0)
std_tot  = np.nanstd(x_tot, axis=0)

ax.plot(r_centers, mean_tot, lw=2.8, c=total_color, ls="-", alpha=1.0, zorder=5)
ax.fill_between(
    r_centers,
    mean_tot - std_tot,
    mean_tot + std_tot,
    color=total_color,
    alpha=0.12,
    zorder=4,
)

# -----------------------
# 2) Plot AUTO means (dotted, colored)
# -----------------------
auto_means = {}
for key in auto_keys:
    x_auto = np.asarray(xi_auto[key])
    mean_auto = np.nanmean(x_auto, axis=0)
    auto_means[key] = mean_auto

    ax.plot(
        r_centers,
        mean_auto,
        lw=2.2,
        c=auto_colors[key],
        ls=":",
        alpha=0.95,
        zorder=3,
    )

# -----------------------
# 3) Plot CROSS means (dashed, colored) + peak markers
# -----------------------
cross_means = []
for (key_i, key_j) in pairs:
    col = cross_colors[(key_i, key_j)]
    ls = "--"

    x_ij = np.asarray(xi_cross[key_i][key_j])
    mean_ij = np.nanmean(x_ij, axis=0)
    cross_means.append(mean_ij)

    ax.plot(
        r_centers,
        mean_ij,
        lw=2.2,
        c=col,
        ls=ls,
        alpha=0.95,
        zorder=2,
    )

    # expected cross-peak marker from rest wavelengths
    lam_i = float(key_to_line[key_i].lambda0_A)
    lam_j = float(key_to_line[key_j].lambda0_A)

    expected_cross_peak = abs(phys.c * 1e-3 * np.log(lam_i / lam_j))
    ax.axvline(
        expected_cross_peak,
        color=col,
        linestyle=ls,
        lw=1.6,
        alpha=0.55,
        zorder=1,
    )

# -----------------------
# 4) Common y-limits
# -----------------------
ymins, ymaxs = [], []

ymins.append(np.nanmin(mean_tot - std_tot))
ymaxs.append(np.nanmax(mean_tot + std_tot))

for key in auto_keys:
    ymins.append(np.nanmin(auto_means[key]))
    ymaxs.append(np.nanmax(auto_means[key]))

for mean_ij in cross_means:
    ymins.append(np.nanmin(mean_ij))
    ymaxs.append(np.nanmax(mean_ij))

ymin = float(np.nanmin(ymins))
ymax = float(np.nanmax(ymaxs))
pad = 0.06 * (ymax - ymin) if np.isfinite(ymax - ymin) and (ymax > ymin) else 1.0
ax.set_ylim(ymin - pad, ymax + pad)

# -----------------------
# 5) Cosmetics
# -----------------------
ax.axhline(0.0, color="k", lw=1.0, alpha=0.35)
ax.set_xlim(r_centers.min(), r_centers.max())
ax.set_xlabel(r"Separation $\Delta v \; [\mathrm{km/s}]$")
ax.set_ylabel(r"$\xi$")

# =======================
# THREE LEGENDS
# =======================

# (1) Linestyle legend
style_handles = [
    Line2D([0], [0], color="k", lw=2.8, ls="-",  label=r"$\xi_\mathrm{T}$"),
    Line2D([0], [0], color="k", lw=2.2, ls=":",  label=r"$\xi_\mathrm{i}$"),
    Line2D([0], [0], color="k", lw=2.2, ls="--", label=r"$\xi_\mathrm{i,j}$"),
]
leg_style = ax.legend(handles=style_handles, loc="upper left", frameon=True, fontsize=16, ncols=3)
ax.add_artist(leg_style)

# (2) Auto legend: total + individual wrapped lines
auto_color_handles = [
    Line2D([0], [0], color=total_color, lw=5.0, ls="-", label=r"$\xi_\mathrm{total}$")
]
for key in auto_keys:
    auto_color_handles.append(
        Line2D(
            [0], [0],
            color=auto_colors[key],
            lw=5.0,
            ls="-",
            label=rf"$\xi_\mathrm{{{key_to_name[key]}}}$",
        )
    )

leg_auto = ax.legend(
    handles=auto_color_handles,
    loc="lower right",
    frameon=True,
    fontsize=16,
    ncols=len(auto_keys) + 1,
)
ax.add_artist(leg_auto)

# (3) Cross legend
cross_color_handles = []
for (key_i, key_j) in pairs:
    cross_color_handles.append(
        Line2D(
            [0], [0],
            color=cross_colors[(key_i, key_j)],
            lw=5.0,
            ls="-",
            label=rf"$\xi_\mathrm{{{key_to_name[key_i]},{key_to_name[key_j]}}}$",
        )
    )

if len(cross_color_handles) > 0:
    leg_cross = ax.legend(
        handles=cross_color_handles,
        loc="upper right",
        frameon=True,
        fontsize=16,
        ncol=1,
    )
    ax.add_artist(leg_cross)

# -----------------------
# 6) Save + show
# -----------------------
savepath_png = os.path.join(save_path_global, "xi1D_wrapped.png")
os.makedirs(os.path.dirname(savepath_png), exist_ok=True)
fig.savefig(savepath_png, dpi=200, bbox_inches="tight")

plt.tight_layout()
plt.show()

# $P1D$